In [1]:
from tqdm import tqdm
import numpy as np
import h5py
import re
import xml.dom.minidom as minidom
import ismrmrd
import xml.etree.ElementTree as ET
import os


os.chdir(os.path.expanduser("~"))
os.chdir(os.path.join(os.getcwd(), "../.."))

data_folder = os.path.join(os.getcwd(), "data/datasets/msk_mri_h5/h5")

print(f"Total files in folder before cleaning: {len(os.listdir(data_folder))}")
DELETE = True


Total files in folder before cleaning: 3292


# Deleting Files by Name

Certain files can be safely ignored or deleted based on their names:

- **Localizer scans**: If the file name contains terms like "LOC", "Localizer", etc., it is a quick scan used only for patient positioning and not a real scan.
- **Adj files**: If the file name contains "Adj" (e.g., `AdjQuietCoilSens`), these are usually not saved or are redundant noise measurements.
- **Phantom/Test scans**: If the file name contains "_sn", it is likely a phantom or test scan.

These files are not useful for further analysis and can be removed to clean the dataset.

In [2]:
files_to_delete = []

MIN_SIZE_MB = 12
total_files = 0
for fname in os.listdir(data_folder):
    fpath = os.path.join(data_folder, fname)
    if os.path.isfile(fpath):
        total_files += 1
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        if size_mb < MIN_SIZE_MB:
            files_to_delete.append((fname, size_mb))

print(f"Total files: {total_files}")
print(f"Files under {MIN_SIZE_MB}MB: {len(files_to_delete)}")
if DELETE: 
    for fname, _ in files_to_delete:
        fpath = os.path.join(data_folder, fname)
        os.remove(fpath)
        print(f"Deleted: {fname}")

pattern_files = [fname for fname in os.listdir(data_folder)
                 if (('Adj' in fname.lower() or '_sn' in fname.lower() or 'loc' in fname.lower()) and os.path.isfile(os.path.join(data_folder, fname)))]
print(f"Files containing 'Adj' or '_sn' or 'loc': {len(pattern_files)}")
if DELETE:
    for fname in pattern_files:
        fpath = os.path.join(data_folder, fname)
        os.remove(fpath)
        print(f"Deleted (Adj/_sn/loc): {fname}")


Total files: 3292
Files under 12MB: 61
Deleted: meas_MID00089_FID115040_SAG_T2_FS.h5
Deleted: meas_MID00290_FID118188_COR_T2_FS.h5
Deleted: meas_MID00078_FID113549_OBLIQUE_SAG_T1_CUFF.h5
Deleted: meas_MID00117_FID115629_RT_SAG_PD_FS.h5
Deleted: meas_MID00263_FID134917_LT_COR_T2_TSE_FS.h5
Deleted: meas_MID00100_FID134754_SAG_T2_FS.h5
Deleted: meas_MID00193_FID118091_SAG_STIR.h5
Deleted: meas_MID00222_FID125348_SAG_T1_IN_OUT_PHASE_DIXON_WEAK.h5
Deleted: meas_MID00098_FID118899_SAG_T2_FS.h5
Deleted: meas_MID00039_FID127088_SAG_DBL_OBL_T2_FS.h5
Deleted: meas_MID00039_FID116510_SAG_T1.h5
Deleted: meas_MID00174_FID135290_LT_AX_T1.h5
Deleted: meas_MID00125_FID115076_SAG_T1.h5
Deleted: meas_MID00191_FID125317_ACR_T1.h5
Deleted: meas_MID00234_FID116231_AX_T1_WP.h5
Deleted: meas_MID00017_FID117915_SAG_T2.h5
Deleted: meas_MID00111_FID129686_SAG_T1.h5
Deleted: meas_MID00261_FID129836_AX_PD_FS.h5
Deleted: meas_MID00053_FID117951_OBL_SAG_T1_FULL.h5
Deleted: meas_MID00044_FID116515_AX_T2_P2.h5
Delete

# Deleting Files by Study Description (`tStudyDescription`)

Some scans are not useful for analysis because they are test, phantom, or quality control scans. These can be identified by specific values in the `tStudyDescription` field of the metadata. If a file's study description matches any of the excluded terms, it should be removed from the dataset.

In [3]:
EXCLUDE_SEQUENCES = [
    # Phantom / QC
    "acr phantom",
    "weekly acr phantom",
    "weekly qc",
    "springbok",

    # Locator / Scout
    "three plane scout",
    "loc",
    "loc uof",
    "cor+-sag loc uof",
    "lt ax loc",
    "rt 3-plane loc uof",
    "l-spine loc uof",
    "haste t-spine loc",

    # Ambiguous / site-specific / test
    "site t2",
    "uof",
    "wp",
    "p2",
    "angle disc",
    "oblique",

    # Generic patterns to catch variations (optional)
    "scout",
    "survey",
    "phantom",
    "qc",
    "localizer",
    "test"
]

all_files = [f for f in os.listdir(data_folder) if os.path.isfile(os.path.join(data_folder, f))]
files_to_delete = []
# ------------------- HELPERS -------------------
def _first_text_from_tag(parent_node, tag_name):
    elems = parent_node.getElementsByTagName(tag_name)
    if elems:
        for node in elems[0].childNodes:
            if node.nodeType == node.TEXT_NODE:
                txt = node.data.strip()
                if txt:
                    return txt
    return None

def read_xml_from_h5(h5_path):
    try:
        with h5py.File(h5_path, "r") as f:
            if "dataset" in f and "xml" in f["dataset"]:
                raw = f["dataset"]["xml"][0]
                if isinstance(raw, (bytes, bytearray)):
                    return raw.decode("utf-8", errors="ignore")
                return str(raw)
    except Exception:
        return None
    return None

def parse_xml_fields(xml_str):
    out = {
        "patientID": None,
        "tStudyDescription": None,
        "patientPosition": None,
        "receiverChannels": None,
        "matrixX": None,
        "matrixY": None,
        "slices": None
    }
    try:
        doc = minidom.parseString(xml_str)
    except Exception:
        return out

    # tStudyDescription
    ups = doc.getElementsByTagName("userParameterString")
    for up in ups:
        name = _first_text_from_tag(up, "name")
        if name == "tStudyDescription":
            out["tStudyDescription"] = _first_text_from_tag(up, "value")
            if out["tStudyDescription"].lower() in EXCLUDE_SEQUENCES:
                files_to_delete.append(fname)
            break

    return out

for fname in all_files:
    xml_str = read_xml_from_h5(os.path.join(data_folder, fname))

print(f"Files to delete by tStudyDescription: {len(files_to_delete)}")

for fname in files_to_delete:
    if DELETE:
        fpath = os.path.join(data_folder, fname)
        os.remove(fpath)
        print(f"Deleted by tStudyDescription: {fname}")

Files to delete by tStudyDescription: 0


# Add Metadata Fields and Delete by Aspect Ratio

To further clean the dataset, we add new fields to the metadata to record the true shape of the k-space. If the aspect ratio of the scan is less than 1/7, it is likely a line scan and can be deleted. Since the original metadata may not contain the k-space shape, we extract and add this information manually.

In [4]:
# ---- Step 1: infer true shape from acquisitions ----
def get_shape_from_ismrmrd(path):
    dset = ismrmrd.Dataset(path, 'dataset', create_if_needed=False)
    n = dset.number_of_acquisitions()
    if n == 0:
        raise ValueError("No acquisitions found.")
    acq0 = dset.read_acquisition(0)
    Nx = int(acq0.number_of_samples)
    nCh = int(acq0.active_channels)

    import numpy as np
    ky = np.zeros(n, dtype=int)
    kz = np.zeros(n, dtype=int)
    sl = np.zeros(n, dtype=int)
    for i in range(n):
        acq = dset.read_acquisition(i)
        md = acq.idx
        ky[i] = getattr(md, 'kspace_encode_step_1', 0)
        kz[i] = getattr(md, 'kspace_encode_step_2', 0)
        sl[i] = getattr(md, 'slice', 0)
    dset.close()

    Ny = int(ky.max() + 1) if ky.size else 1
    Nz = int(kz.max() + 1) if kz.size else 1
    Nslices = int(sl.max() + 1) if sl.size else 1
    S = Nslices * Nz
    return dict(Nx=Nx, Ny=Ny, Nz=Nz, Nslices=Nslices, S=S, Channels=nCh, H=Ny, W=Nx, C=nCh)

# ---- Step 2: append new userParameters entries (do not replace existing fields) ----
def _detect_namespace(root):
    m = re.match(r'^\{(.*)\}', root.tag)
    return m.group(1) if m else None

def _q(tag, ns):
    return f"{{{ns}}}{tag}" if ns else tag

def _ensure(parent, tag, ns):
    node = parent.find(_q(tag, ns))
    if node is None:
        node = ET.SubElement(parent, _q(tag, ns))
    return node

def _add_user_param_string(user_params, ns, name, value):
    ups = ET.SubElement(user_params, _q("userParameterString", ns))
    n = ET.SubElement(ups, _q("name", ns)); n.text = str(name)
    v = ET.SubElement(ups, _q("value", ns)); v.text = str(value)

def add_real_shape_userparams(path, dims):
    """
    Appends (doesn't replace) the following under userParameters:
      RealNx, RealNy, RealNz, RealNslices, RealS, RealChannels, RealShape_S_C_H_W_2
    Leaves matrixSize/reconSpace/etc. exactly as-is.
    """
    with h5py.File(path, "r+") as f:
        xml_ds = f["dataset"]["xml"]
        raw = xml_ds[0]
        xml_str = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else str(raw)

        root = ET.fromstring(xml_str)
        ns = _detect_namespace(root)
        if ns:
            ET.register_namespace("", ns)

        user_params = _ensure(root, "userParameters", ns)

        _add_user_param_string(user_params, ns, "Nx", dims["Nx"])
        _add_user_param_string(user_params, ns, "Ny", dims["Ny"])

        new_xml = ET.tostring(root, encoding="utf-8").decode("utf-8")

        # one-time backup
        if "xml_backup_before_shape_fix" not in f["dataset"]:
            f["dataset"].create_dataset("xml_backup_before_shape_fix", data=np.string_(xml_str))

        try:
            xml_ds[0] = new_xml
        except Exception:
            del f["dataset"]["xml"]
            vlen_utf8 = h5py.string_dtype("utf-8")
            f["dataset"].create_dataset("xml", data=new_xml, dtype=vlen_utf8)

selected_files = []

def annotate_file_with_real_shape(path):
    dims = get_shape_from_ismrmrd(path)

    ar = min(dims['H']/dims['W'], dims['W']/dims['H'])
    if ar < 1/7:
        selected_files.append(path)

    add_real_shape_userparams(path, dims)

# --- Gather files and process only those not starting with "clean_" ---
file_list = [
    os.path.join(data_folder, f)
    for f in os.listdir(data_folder)
    if os.path.isfile(os.path.join(data_folder, f)) and f.endswith(".h5")
]

for fpath in tqdm(file_list, desc="Processing files", unit="file"):
    fname = os.path.basename(fpath)

    # Only change files that do NOT start with "clean_"
    if fname.startswith("clean_"):
        continue

    # Build renamed path
    new_name = "clean_" + fname
    new_path = os.path.join(os.path.dirname(fpath), new_name)

    try:
        # Rename first
        os.rename(fpath, new_path)
    except FileExistsError:
        # If a file with the target name already exists, skip safely
        print(f"Skipped rename (exists): {new_name}")
        continue
    except Exception as e:
        print(f"Error renaming {fname}: {e}")
        continue

    try:
        annotate_file_with_real_shape(new_path)
    except Exception as e:
        print(f"Error processing file {new_name}: {e}")

print("\nSelected files:", selected_files)

if DELETE:
    for fname in files_to_delete:
        fpath = os.path.join(data_folder, fname)
        os.remove(fpath)
        print(f"Deleted by high aspect ratio: {fname}")


Processing files: 100%|██████████| 1671/1671 [00:00<00:00, 1891171.61file/s]


Selected files: []


## Delete mismatched `.dat` files (do not run this)

In [ ]:
h5_dir = "data/datasets/msk_mri_h5/h5"
dat_dir = "data/datasets/msk_mri"

clean = lambda f: os.path.splitext(f)[0].removeprefix("clean_")

h5_names = {clean(f) for f in os.listdir(h5_dir)}
dat_files = os.listdir(dat_dir)

for f in dat_files:
    base = clean(f)
    if base not in h5_names and f.endswith(".dat"):
        path = os.path.join(dat_dir, f)
        os.remove(path)
        print(f" Deleted: {f}")

print("Done deleting mismatched .dat files.")


 Deleted: meas_MID00157_FID132350_Three_Plane_Scout.dat
Done deleting mismatched .dat files.
